# 📊 RecruitIQ Model Evaluation Benchmarks
This notebook evaluates the trained LightGBM ranker (`model/model.pkl`) using unseen synthetic test queries. It calculates ranking accuracy using the **NDCG (Normalized Discounted Cumulative Gain)** metric at various cut-offs ($K=1, 3, 5, 10$).

In [1]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
from sklearn.metrics import ndcg_score

# Ensure project root is in path
sys.path.append(os.path.abspath('.'))

## 1. Load the Model
We load the trained LightGBM ranker model from `model/model.pkl`.

In [2]:
model_path = os.path.join("model", "model.pkl")
print(f"Loading model from {model_path}...")
model = joblib.load(model_path)
print("Model loaded successfully!")

## 2. Generate Unseen Test Data
We generate 50 evaluation queries (with 20 candidates per query) using a different random seed (`seed=100`) to guarantee that the test data was not seen during model training.

In [3]:
def generate_test_data(num_queries=50, candidates_per_query=20, seed=100):
    features_columns = [
        'skill_overlap_score', 'education_match_score', 'responsibility_similarity_score', 
        'language_match_score', 'certification_match_score', 'experience_years_score',
        'projects_count_score', 'major_match_score', 'seniority_match_score',
        'skill_breadth_score', 'job_stability_score', 'online_presence_score', 'responsibility_depth_score',
        'core_skill_coverage', 'skill_relevance_score', 'skill_context_score', 'skill_frequency_score',
        'rare_skill_bonus', 'skill_recency_score', 'skill_group_match_score', 'skill_depth_score',
        'experience_relevance_score', 'role_progression_score', 'role_similarity_score', 'company_relevance_score',
        'experience_gap_penalty', 'leadership_experience_score', 'role_duration_consistency',
        'responsibility_alignment_score', 'responsibility_complexity_score', 'impact_score',
        'action_verb_density', 'responsibility_diversity_score',
        'degree_level_score', 'education_relevance_score', 'academic_performance_score', 'institution_tier_score',
        'project_relevance_score', 'project_complexity_score', 'project_impact_score', 'project_recency_score',
        'certification_relevance_score', 'certification_authority_score', 'certification_recency_score',
        'communication_score', 'initiative_score', 'leadership_signal_score',
        'resume_jd_embedding_score', 'skill_embedding_match_score', 'experience_embedding_score'
    ]
    
    total_samples = num_queries * candidates_per_query
    np.random.seed(seed)
    X = pd.DataFrame(np.random.rand(total_samples, len(features_columns)), columns=features_columns)
    
    # Calculate target based on key features + noise
    raw_target = X['skill_overlap_score'] * 0.4 + X['experience_years_score'] * 0.4 + np.random.rand(total_samples) * 0.2
    y = pd.cut(raw_target, bins=[-np.inf, 0.4, 0.7, np.inf], labels=[0, 1, 2]).astype(int)
    
    return X, y

num_queries = 50
candidates_per_query = 20
X_test, y_test = generate_test_data(num_queries=num_queries, candidates_per_query=candidates_per_query, seed=100)
print(f"Generated {num_queries} queries with {candidates_per_query} candidates each. Total samples: {len(X_test)}")

## 3. Run Inference & Rank Candidates
We run the LightGBM ranker on the test features to predict ranking scores.

In [4]:
preds = model.predict(X_test)
print("Inference completed on the test set.")

## 4. Compute NDCG Evaluation Metrics
We compute NDCG at thresholds $K=1, 3, 5, 10$ across all test queries.

In [5]:
ndcg_1_list = []
ndcg_3_list = []
ndcg_5_list = []
ndcg_10_list = []

for q_idx in range(num_queries):
    start = q_idx * candidates_per_query
    end = start + candidates_per_query
    
    y_true = np.array(y_test.iloc[start:end])
    y_pred = preds[start:end]
    
    if np.sum(y_true) == 0:
        continue
        
    ndcg_1 = ndcg_score([y_true], [y_pred], k=1)
    ndcg_3 = ndcg_score([y_true], [y_pred], k=3)
    ndcg_5 = ndcg_score([y_true], [y_pred], k=5)
    ndcg_10 = ndcg_score([y_true], [y_pred], k=10)
    
    ndcg_1_list.append(ndcg_1)
    ndcg_3_list.append(ndcg_3)
    ndcg_5_list.append(ndcg_5)
    ndcg_10_list.append(ndcg_10)

mean_ndcg_1 = np.mean(ndcg_1_list)
mean_ndcg_3 = np.mean(ndcg_3_list)
mean_ndcg_5 = np.mean(ndcg_5_list)
mean_ndcg_10 = np.mean(ndcg_10_list)

print("\n--- Model Evaluation Metrics (Unseen Test Data) ---")
print(f"Mean NDCG@1:  {mean_ndcg_1:.4f}")
print(f"Mean NDCG@3:  {mean_ndcg_3:.4f}")
print(f"Mean NDCG@5:  {mean_ndcg_5:.4f}")
print(f"Mean NDCG@10: {mean_ndcg_10:.4f}")
print("--------------------------------------------------")


--- Model Evaluation Metrics (Unseen Test Data) ---
Mean NDCG@1:  0.8800
Mean NDCG@3:  0.9228
Mean NDCG@5:  0.9451
Mean NDCG@10: 0.9618
--------------------------------------------------
